<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 2 - Claudia

## Validate source filename sequence

In [1]:
from pathlib import Path
import re

source_dir = Path("corpus/00_sources/00_original")

filename_pattern = re.compile(r"^(\d{3}|1000)\.txt$")

files = sorted(source_dir.glob("*.txt"), key=lambda path: int(path.stem))
filenames = [path.name for path in files]

invalid_filenames = [
    name for name in filenames
    if not filename_pattern.fullmatch(name)
]

parsed_numbers = [
    int(path.stem)
    for path in files
    if filename_pattern.fullmatch(path.name)
]

expected_numbers = list(range(1, 1001))
expected_filenames = [
    f"{number:03d}.txt" if number < 1000 else "1000.txt"
    for number in expected_numbers
]

missing_filenames = sorted(set(expected_filenames) - set(filenames))
extra_filenames = sorted(set(filenames) - set(expected_filenames))

is_uninterrupted = (
        parsed_numbers == expected_numbers
        and not invalid_filenames
        and not missing_filenames
        and not extra_filenames
)

print(f"Files found: {len(filenames)}")
print(f"First file: {filenames[0] if filenames else 'None'}")
print(f"Last file: {filenames[-1] if filenames else 'None'}")
print(f"Invalid filenames: {invalid_filenames or 'None'}")
print(f"Missing filenames: {missing_filenames or 'None'}")
print(f"Extra filenames: {extra_filenames or 'None'}")
print(f"Uninterrupted 001.txt–1000.txt sequence: {is_uninterrupted}")

assert is_uninterrupted, "Source filenames are not an uninterrupted sequence from 001.txt to 1000.txt."

Files found: 1000
First file: 001.txt
Last file: 1000.txt
Invalid filenames: None
Missing filenames: None
Extra filenames: None
Uninterrupted 001.txt–1000.txt sequence: True


## Recover original transcript IDs for numbered source files

In [2]:
from collections import defaultdict
from hashlib import sha256
from pathlib import Path

import pandas as pd

source_dir = Path("corpus/00_sources/00_original")
original_transcripts_dir = Path("../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini")
mapping_output_path = Path("corpus/00_sources/00_sources_mapping.tsv")


def file_sha256(path: Path) -> str:
    """Return the SHA-256 hash of a file's raw bytes."""
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_files = sorted(source_dir.glob("*.txt"), key=lambda path: int(path.stem))
original_files = sorted(original_transcripts_dir.glob("*.txt"))

original_files_by_hash = defaultdict(list)
for original_file in original_files:
    original_files_by_hash[file_sha256(original_file)].append(original_file)

mapping_rows = []
for source_file in source_files:
    source_hash = file_sha256(source_file)
    matching_original_files = original_files_by_hash.get(source_hash, [])
    matching_original_names = [path.name for path in matching_original_files]
    matching_corpus_ids = [path.stem for path in matching_original_files]

    if len(matching_original_files) == 1:
        status = "matched"
    elif len(matching_original_files) == 0:
        status = "unmatched"
    else:
        status = "ambiguous"

    mapping_rows.append(
        {
            "source_number": int(source_file.stem),
            "source_file": source_file.as_posix(),
            "matched_original_file": matching_original_files[0].as_posix() if len(matching_original_files) == 1 else None,
            "corpus_id": matching_corpus_ids[0] if len(matching_corpus_ids) == 1 else None,
            "match_count": len(matching_original_files),
            "status": status,
            "all_matching_original_files": "; ".join(matching_original_names),
            "sha256": source_hash,
        }
    )

source_mapping_df = pd.DataFrame(mapping_rows)
source_mapping_df.to_csv(mapping_output_path, sep="	", index=False)

status_counts = source_mapping_df["status"].value_counts().to_dict()
print(f"Phase 2 source files: {len(source_files)}")
print(f"Phase 1 candidate files: {len(original_files)}")
print(f"Matched uniquely: {status_counts.get('matched', 0)}")
print(f"Unmatched: {status_counts.get('unmatched', 0)}")
print(f"Ambiguous: {status_counts.get('ambiguous', 0)}")
print(f"Mapping written to: {mapping_output_path}")

source_mapping_df

Phase 2 source files: 1000
Phase 1 candidate files: 1000
Matched uniquely: 999
Unmatched: 1
Ambiguous: 0
Mapping written to: corpus/00_sources/00_sources_mapping.tsv


,source_number,source_file,matched_original_file,corpus_id,match_count,status,all_matching_original_files,sha256
0,1,corpus/00_sources/00_original/001.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,eH3giaIzONA,1,matched,eH3giaIzONA.txt,408fb3d07b5a90f22dc0c27c5009740d9d12fda275f303...
1,2,corpus/00_sources/00_original/002.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,xFrGuyw1V8s,1,matched,xFrGuyw1V8s.txt,b55da182ebea1985d0bdd7022760412363449d11588d7a...
2,3,corpus/00_sources/00_original/003.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,y3KJ7d2qBoA,1,matched,y3KJ7d2qBoA.txt,0a40c55c88477b2d6492f689143a61bab3ed487d1946cd...
3,4,corpus/00_sources/00_original/004.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,4fndeDfaWCg,1,matched,4fndeDfaWCg.txt,d9e8ef74e67b34e2d4498664dc8216a148667954087daa...
4,5,corpus/00_sources/00_original/005.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,R7UrFYvl5TE,1,matched,R7UrFYvl5TE.txt,e38e1822a047a94c8b6240d83e2d917c54f2cde3a685d6...
...,...,...,...,...,...,...,...,...
995,996,corpus/00_sources/00_original/996.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,yKNxeF4KMsY,1,matched,yKNxeF4KMsY.txt,aa94b4d14fb5a8d8e461e286ce8dc200d257a2f7b7f8cb...
996,997,corpus/00_sources/00_original/997.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,V1Pl8CzNzCw,1,matched,V1Pl8CzNzCw.txt,692e8ed96b40aae490445f3523992166a704b51d4620d6...
997,998,corpus/00_sources/00_original/998.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,mRD0-GxqHVo,1,matched,mRD0-GxqHVo.txt,2044c3a247b15530a3d09151bc86db94a59f1515e75fe9...
998,999,corpus/00_sources/00_original/999.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,nyuo9-OjNNg,1,matched,nyuo9-OjNNg.txt,7f031bed3d7d6e70b93a9cad25b3b4d8a35c989842a803...


## Inspect unmatched source mappings

In [3]:
unmatched_source_mapping_df = source_mapping_df[
    source_mapping_df["status"].eq("unmatched")
].copy()

print(f"Unmatched rows: {len(unmatched_source_mapping_df)}")

unmatched_source_mapping_df

Unmatched rows: 1


,source_number,source_file,matched_original_file,corpus_id,match_count,status,all_matching_original_files,sha256
248,249,corpus/00_sources/00_original/249.txt,NaN,NaN,0,unmatched,,ad03b4517d5e8db83205bb07be6e5c3b64807beab1c61c...


`corpus/00_sources/00_original/249.txt` was found unmatched. It was copied to `corpus/00_sources/01_unmached/249.txt`. Its content was:

```
Cruzeiro - aluna Sarah
Carapicuiba - alunas Sara e Raine
Santana de Paranaíba - aluno Kayky
Cotia - em edição
Americana - aluna Maira
Ferraz - aluna Patricia (incluir a conta)
São Roque
Barueri
Mogi das Cruzes - aluno Uchida


```

## Duplicate matched original files

In [4]:
# Duplicate matched original files

duplicate_matched_original_files_df = (
    source_mapping_df[
        source_mapping_df["matched_original_file"].notna()
        & source_mapping_df["matched_original_file"].duplicated(keep=False)
        ]
    .sort_values("matched_original_file")
)

duplicate_matched_original_files_df

,source_number,source_file,matched_original_file,corpus_id,match_count,status,all_matching_original_files,sha256
724,725,corpus/00_sources/00_original/725.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,N1MW6xXjW8g,1,matched,N1MW6xXjW8g.txt,489bcb4673e00054bc705c8e91facfc56717c8947fe126...
918,919,corpus/00_sources/00_original/919.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,N1MW6xXjW8g,1,matched,N1MW6xXjW8g.txt,489bcb4673e00054bc705c8e91facfc56717c8947fe126...
137,138,corpus/00_sources/00_original/138.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,beINamVRGy4,1,matched,beINamVRGy4.txt,bad3c7e4d6bc8120f9bb0892b3700b0a482cb953552a97...
501,502,corpus/00_sources/00_original/502.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,beINamVRGy4,1,matched,beINamVRGy4.txt,bad3c7e4d6bc8120f9bb0892b3700b0a482cb953552a97...
273,274,corpus/00_sources/00_original/274.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,i8Whtxcw8oU,1,matched,i8Whtxcw8oU.txt,236f7f418116e7b4c22fefc31d0aaa75ff5ff5c0351685...
392,393,corpus/00_sources/00_original/393.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,i8Whtxcw8oU,1,matched,i8Whtxcw8oU.txt,236f7f418116e7b4c22fefc31d0aaa75ff5ff5c0351685...
135,136,corpus/00_sources/00_original/136.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,mrudT410TAI,1,matched,mrudT410TAI.txt,d10a1dc048abc7e840e6b21de99f680250152036d0a189...
867,868,corpus/00_sources/00_original/868.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,mrudT410TAI,1,matched,mrudT410TAI.txt,d10a1dc048abc7e840e6b21de99f680250152036d0a189...
468,469,corpus/00_sources/00_original/469.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,zR5_KkxgGqE,1,matched,zR5_KkxgGqE.txt,43ad2437426583d7fb905805fc5ef84b405fc3972398dc...
544,545,corpus/00_sources/00_original/545.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,zR5_KkxgGqE,1,matched,zR5_KkxgGqE.txt,43ad2437426583d7fb905805fc5ef84b405fc3972398dc...


The duplicates were copied to `corpus/00_sources/02_duplicate/`

## Unmatched Gemini transcript files

In [5]:
# Unmatched Gemini transcript files

from pathlib import Path

gemini_transcripts_dir = Path("../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini")

matched_original_files = set(source_mapping_df["matched_original_file"].dropna())

unmatched_gemini_transcript_files = sorted(
    str(path)
    for path in gemini_transcripts_dir.glob("*.txt")
    if str(path) not in matched_original_files
)

unmatched_gemini_transcript_files

['../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/-CCgDvUM4TM.txt',
 '../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/ZSM3w1v-A_Y.txt',
 '../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/bQRzrnH6_HY.txt',
 '../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/dzIUgS3Imu0.txt',
 '../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/nagjPyNMu9Q.txt',
 '../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini/oKOtzIo-uYw.txt']

The unmatched files were copied to `corpus/00_sources/03_unmatched_transcript_files/`

## Corrected match

The match was manually corrected, resulting in the following table.

| file_id |          id |
|--------:|------------:|
|     725 | dzIUgS3Imu0 |
|     502 | ZSM3w1v-A_Y |
|     393 | nagjPyNMu9Q |
|     136 | oKOtzIo-uYw |
|     469 | bQRzrnH6_HY |
|     249 | -CCgDvUM4TM |

## Recover original transcript IDs for the corrected numbered source files

In [6]:
from collections import defaultdict
from hashlib import sha256
from pathlib import Path

import pandas as pd

source_dir = Path("corpus/00_sources/05_corrected")
original_transcripts_dir = Path("../cl_st1_ph1_claudia/corpus/02_music_videos_transcripts_gemini")
mapping_output_path = Path("corpus/00_sources/00_sources_mapping_corrected.tsv")


def file_sha256(path: Path) -> str:
    """Return the SHA-256 hash of a file's raw bytes."""
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_files = sorted(source_dir.glob("*.txt"), key=lambda path: int(path.stem))
original_files = sorted(original_transcripts_dir.glob("*.txt"))

original_files_by_hash = defaultdict(list)
for original_file in original_files:
    original_files_by_hash[file_sha256(original_file)].append(original_file)

mapping_rows = []
for source_file in source_files:
    source_hash = file_sha256(source_file)
    matching_original_files = original_files_by_hash.get(source_hash, [])
    matching_original_names = [path.name for path in matching_original_files]
    matching_corpus_ids = [path.stem for path in matching_original_files]

    if len(matching_original_files) == 1:
        status = "matched"
    elif len(matching_original_files) == 0:
        status = "unmatched"
    else:
        status = "ambiguous"

    mapping_rows.append(
        {
            "source_number": int(source_file.stem),
            "source_file": source_file.as_posix(),
            "matched_original_file": matching_original_files[0].as_posix() if len(matching_original_files) == 1 else None,
            "corpus_id": matching_corpus_ids[0] if len(matching_corpus_ids) == 1 else None,
            "match_count": len(matching_original_files),
            "status": status,
            "all_matching_original_files": "; ".join(matching_original_names),
            "sha256": source_hash,
        }
    )

source_mapping_df = pd.DataFrame(mapping_rows)
source_mapping_df.to_csv(mapping_output_path, sep="	", index=False)

status_counts = source_mapping_df["status"].value_counts().to_dict()
print(f"Phase 2 source files: {len(source_files)}")
print(f"Phase 1 candidate files: {len(original_files)}")
print(f"Matched uniquely: {status_counts.get('matched', 0)}")
print(f"Unmatched: {status_counts.get('unmatched', 0)}")
print(f"Ambiguous: {status_counts.get('ambiguous', 0)}")
print(f"Mapping written to: {mapping_output_path}")

source_mapping_df

Phase 2 source files: 1000
Phase 1 candidate files: 1000
Matched uniquely: 1000
Unmatched: 0
Ambiguous: 0
Mapping written to: corpus/00_sources/00_sources_mapping_corrected.tsv


,source_number,source_file,matched_original_file,corpus_id,match_count,status,all_matching_original_files,sha256
0,1,corpus/00_sources/05_corrected/001.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,eH3giaIzONA,1,matched,eH3giaIzONA.txt,408fb3d07b5a90f22dc0c27c5009740d9d12fda275f303...
1,2,corpus/00_sources/05_corrected/002.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,xFrGuyw1V8s,1,matched,xFrGuyw1V8s.txt,b55da182ebea1985d0bdd7022760412363449d11588d7a...
2,3,corpus/00_sources/05_corrected/003.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,y3KJ7d2qBoA,1,matched,y3KJ7d2qBoA.txt,0a40c55c88477b2d6492f689143a61bab3ed487d1946cd...
3,4,corpus/00_sources/05_corrected/004.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,4fndeDfaWCg,1,matched,4fndeDfaWCg.txt,d9e8ef74e67b34e2d4498664dc8216a148667954087daa...
4,5,corpus/00_sources/05_corrected/005.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,R7UrFYvl5TE,1,matched,R7UrFYvl5TE.txt,e38e1822a047a94c8b6240d83e2d917c54f2cde3a685d6...
...,...,...,...,...,...,...,...,...
995,996,corpus/00_sources/05_corrected/996.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,yKNxeF4KMsY,1,matched,yKNxeF4KMsY.txt,aa94b4d14fb5a8d8e461e286ce8dc200d257a2f7b7f8cb...
996,997,corpus/00_sources/05_corrected/997.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,V1Pl8CzNzCw,1,matched,V1Pl8CzNzCw.txt,692e8ed96b40aae490445f3523992166a704b51d4620d6...
997,998,corpus/00_sources/05_corrected/998.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,mRD0-GxqHVo,1,matched,mRD0-GxqHVo.txt,2044c3a247b15530a3d09151bc86db94a59f1515e75fe9...
998,999,corpus/00_sources/05_corrected/999.txt,../cl_st1_ph1_claudia/corpus/02_music_videos_t...,nyuo9-OjNNg,1,matched,nyuo9-OjNNg.txt,7f031bed3d7d6e70b93a9cad25b3b4d8a35c989842a803...


## Load and enrich music video dataset with source file IDs

In [7]:
from pathlib import Path

import pandas as pd

boolean_columns = ["within_iqr", "within_non_outlier_range"]

df_music_videos_dataset = pd.read_json(
    "../cl_st1_ph1_claudia/corpus/00_sources/music_videos_dataset.ndjson",
    lines=True,
)

for column in boolean_columns:
    df_music_videos_dataset[column] = df_music_videos_dataset[column].astype("boolean")

df_sources_mapping_corrected = pd.read_csv(
    "corpus/00_sources/00_sources_mapping_corrected.tsv",
    sep="\t",
)

df_file_ids = (
    df_sources_mapping_corrected[["corpus_id", "source_file"]]
    .assign(file_id=lambda df: df["source_file"].map(lambda path: Path(path).stem))
    .rename(columns={"corpus_id": "id"})
    [["id", "file_id"]]
)

df_music_videos_dataset = df_file_ids.merge(
    df_music_videos_dataset,
    on="id",
    how="right",
)

df_music_videos_dataset = df_music_videos_dataset[
    ["file_id", *[column for column in df_music_videos_dataset.columns if column != "file_id"]]
]

df_music_videos_dataset

,file_id,id,url,title,description,language,uploader,duration,upload_date,transcript_word_count,within_iqr,within_non_outlier_range,available
0,1000,4xqo7D2k8HM,https://www.youtube.com/watch?v=4xqo7D2k8HM,Imagine Dragons - Believer (Live in Vegas),Watch Imagine Dragons Live in Vegas on Hulu: \...,NaN,ImagineDragons,280.0,20230720.0,407.0,False,True,True
1,999,nyuo9-OjNNg,https://www.youtube.com/watch?v=nyuo9-OjNNg,I Wanna Be Yours,Provided to YouTube by Domino\n\nI Wanna Be Yo...,NaN,Official Arctic Monkeys,184.0,20141209.0,228.0,True,True,True
2,998,mRD0-GxqHVo,https://www.youtube.com/watch?v=mRD0-GxqHVo,Glass Animals - Heat Waves (Official Video),I LOVE YOU SO F***ING MUCH.\nOur fourth studio...,NaN,Glass Animals,235.0,20200629.0,389.0,False,True,True
3,997,V1Pl8CzNzCw,https://www.youtube.com/watch?v=V1Pl8CzNzCw,"Billie Eilish, Khalid - lovely",Listen to “lovely” (with Khalid): http://smart...,NaN,Billie Eilish,201.0,20180426.0,176.0,False,True,True
4,996,yKNxeF4KMsY,https://www.youtube.com/watch?v=yKNxeF4KMsY,Coldplay - Yellow (Official Video),Coldplay's tenth studio album Moon Music avail...,NaN,Coldplay,272.0,20110530.0,200.0,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006,621,Er9xGRolrT4,https://www.youtube.com/watch?v=Er9xGRolrT4,Chic - Good Times (Atlantic Records 1979),"""Good Times"" is a 1979 song composed by Bernar...",en,DiscoSaturdayNightTV,503.0,20141129.0,359.0,True,True,True
1007,549,Wxx_2q409oY,https://www.youtube.com/watch?v=Wxx_2q409oY,Vertical Horizon - Everything You Want 1999,Vertical Horizon - Everything You Want 1999\nR...,NaN,Melopsittacus undulatus,259.0,20160619.0,330.0,True,True,True
1008,494,UVwKxz0e2sE,https://www.youtube.com/watch?v=UVwKxz0e2sE,Tommy Tutone - 867-5309 / Jenny (Lyrics HD),"Listen to ""867-5309 / Jenny"" by Tommy Tutone, ...",NaN,Song Lyrics HD,236.0,20210217.0,186.0,False,True,True
1009,400,j2F4INQFjEI,https://www.youtube.com/watch?v=j2F4INQFjEI,Belinda Carlisle - Heaven Is A Place On Earth ...,Official 4K Remastered Music Video for ‘Heaven...,NaN,Belinda Carlisle,244.0,20130522.0,304.0,True,True,True


## Sort Music Videos by `file_id`

In [8]:
df_music_videos_dataset = df_music_videos_dataset.sort_values(
    by="file_id",
    ascending=False
)
df_music_videos_dataset

,file_id,id,url,title,description,language,uploader,duration,upload_date,transcript_word_count,within_iqr,within_non_outlier_range,available
1,999,nyuo9-OjNNg,https://www.youtube.com/watch?v=nyuo9-OjNNg,I Wanna Be Yours,Provided to YouTube by Domino\n\nI Wanna Be Yo...,NaN,Official Arctic Monkeys,184.0,20141209.0,228.0,True,True,True
2,998,mRD0-GxqHVo,https://www.youtube.com/watch?v=mRD0-GxqHVo,Glass Animals - Heat Waves (Official Video),I LOVE YOU SO F***ING MUCH.\nOur fourth studio...,NaN,Glass Animals,235.0,20200629.0,389.0,False,True,True
3,997,V1Pl8CzNzCw,https://www.youtube.com/watch?v=V1Pl8CzNzCw,"Billie Eilish, Khalid - lovely",Listen to “lovely” (with Khalid): http://smart...,NaN,Billie Eilish,201.0,20180426.0,176.0,False,True,True
4,996,yKNxeF4KMsY,https://www.youtube.com/watch?v=yKNxeF4KMsY,Coldplay - Yellow (Official Video),Coldplay's tenth studio album Moon Music avail...,NaN,Coldplay,272.0,20110530.0,200.0,False,True,True
5,995,KtlgYxa6BMU,https://www.youtube.com/watch?v=KtlgYxa6BMU,Lord Huron - The Night We Met (Official Audio),"Order Strange Trails featuring ""The Night We M...",NaN,Lord Huron,208.0,20150209.0,174.0,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,NaN,NaN,https://www.youtube.com/watch?v=c3Hqy88oLIg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,False
450,NaN,NaN,https://www.youtube.com/watch?v=rZoD8JEFjAE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,False
503,NaN,NaN,https://www.youtube.com/watch?v=6WTdTwcmxyo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,False
597,NaN,NaN,https://www.youtube.com/watch?v=NOGEyBeoBGM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,False


## Extract upload year from upload date

In [9]:
# Extract the year from upload_date and place it immediately after upload_date
upload_date_position = df_music_videos_dataset.columns.get_loc("upload_date") + 1

year = (
    pd.to_numeric(df_music_videos_dataset["upload_date"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str[:4]
    .astype("Int64")
)

if "year" in df_music_videos_dataset.columns:
    df_music_videos_dataset = df_music_videos_dataset.drop(columns="year")

df_music_videos_dataset.insert(upload_date_position, "year", year)

df_music_videos_dataset

,file_id,id,url,title,description,language,uploader,duration,upload_date,year,transcript_word_count,within_iqr,within_non_outlier_range,available
1,999,nyuo9-OjNNg,https://www.youtube.com/watch?v=nyuo9-OjNNg,I Wanna Be Yours,Provided to YouTube by Domino\n\nI Wanna Be Yo...,NaN,Official Arctic Monkeys,184.0,20141209.0,2014,228.0,True,True,True
2,998,mRD0-GxqHVo,https://www.youtube.com/watch?v=mRD0-GxqHVo,Glass Animals - Heat Waves (Official Video),I LOVE YOU SO F***ING MUCH.\nOur fourth studio...,NaN,Glass Animals,235.0,20200629.0,2020,389.0,False,True,True
3,997,V1Pl8CzNzCw,https://www.youtube.com/watch?v=V1Pl8CzNzCw,"Billie Eilish, Khalid - lovely",Listen to “lovely” (with Khalid): http://smart...,NaN,Billie Eilish,201.0,20180426.0,2018,176.0,False,True,True
4,996,yKNxeF4KMsY,https://www.youtube.com/watch?v=yKNxeF4KMsY,Coldplay - Yellow (Official Video),Coldplay's tenth studio album Moon Music avail...,NaN,Coldplay,272.0,20110530.0,2011,200.0,False,True,True
5,995,KtlgYxa6BMU,https://www.youtube.com/watch?v=KtlgYxa6BMU,Lord Huron - The Night We Met (Official Audio),"Order Strange Trails featuring ""The Night We M...",NaN,Lord Huron,208.0,20150209.0,2015,174.0,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,NaN,NaN,https://www.youtube.com/watch?v=c3Hqy88oLIg,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,False
450,NaN,NaN,https://www.youtube.com/watch?v=rZoD8JEFjAE,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,False
503,NaN,NaN,https://www.youtube.com/watch?v=6WTdTwcmxyo,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,False
597,NaN,NaN,https://www.youtube.com/watch?v=NOGEyBeoBGM,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,False


## Create the `decade` column based on the `year` column

In [10]:
# Create the decade column and place it immediately after year
year_position = df_music_videos_dataset.columns.get_loc("year") + 1

decade = (df_music_videos_dataset["year"] // 10 * 10).astype("Int64")

if "decade" in df_music_videos_dataset.columns:
    df_music_videos_dataset = df_music_videos_dataset.drop(columns="decade")

df_music_videos_dataset.insert(year_position, "decade", decade)

df_music_videos_dataset

,file_id,id,url,title,description,language,uploader,duration,upload_date,year,decade,transcript_word_count,within_iqr,within_non_outlier_range,available
1,999,nyuo9-OjNNg,https://www.youtube.com/watch?v=nyuo9-OjNNg,I Wanna Be Yours,Provided to YouTube by Domino\n\nI Wanna Be Yo...,NaN,Official Arctic Monkeys,184.0,20141209.0,2014,2010,228.0,True,True,True
2,998,mRD0-GxqHVo,https://www.youtube.com/watch?v=mRD0-GxqHVo,Glass Animals - Heat Waves (Official Video),I LOVE YOU SO F***ING MUCH.\nOur fourth studio...,NaN,Glass Animals,235.0,20200629.0,2020,2020,389.0,False,True,True
3,997,V1Pl8CzNzCw,https://www.youtube.com/watch?v=V1Pl8CzNzCw,"Billie Eilish, Khalid - lovely",Listen to “lovely” (with Khalid): http://smart...,NaN,Billie Eilish,201.0,20180426.0,2018,2010,176.0,False,True,True
4,996,yKNxeF4KMsY,https://www.youtube.com/watch?v=yKNxeF4KMsY,Coldplay - Yellow (Official Video),Coldplay's tenth studio album Moon Music avail...,NaN,Coldplay,272.0,20110530.0,2011,2010,200.0,False,True,True
5,995,KtlgYxa6BMU,https://www.youtube.com/watch?v=KtlgYxa6BMU,Lord Huron - The Night We Met (Official Audio),"Order Strange Trails featuring ""The Night We M...",NaN,Lord Huron,208.0,20150209.0,2015,2010,174.0,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,NaN,NaN,https://www.youtube.com/watch?v=c3Hqy88oLIg,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,False
450,NaN,NaN,https://www.youtube.com/watch?v=rZoD8JEFjAE,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,False
503,NaN,NaN,https://www.youtube.com/watch?v=6WTdTwcmxyo,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,False
597,NaN,NaN,https://www.youtube.com/watch?v=NOGEyBeoBGM,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,False


## Set of decades

In [11]:
set(df_music_videos_dataset["decade"].dropna().sort_values().unique())

{np.int64(2000), np.int64(2010), np.int64(2020)}

In [12]:
df_music_videos_dataset["decade"].dropna().value_counts().sort_index()

decade
2000    270
2010    539
2020    191
Name: count, dtype: Int64

## Set of years

In [13]:
set(df_music_videos_dataset["year"].dropna().sort_values().unique())

{np.int64(2006),
 np.int64(2007),
 np.int64(2008),
 np.int64(2009),
 np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024),
 np.int64(2025),
 np.int64(2026)}

In [14]:
df_music_videos_dataset["year"].dropna().value_counts().sort_index()

year
2006      4
2007     12
2008     23
2009    231
2010     68
2011     57
2012     53
2013     57
2014     38
2015     60
2016     44
2017     61
2018     58
2019     43
2020     58
2021     47
2022     35
2023     19
2024     20
2025      6
2026      6
Name: count, dtype: Int64

## Set of uploaders

In [15]:
set(df_music_videos_dataset["uploader"].dropna().sort_values().unique())

{'*NSYNC',
 '05vs1 ',
 '1984dreaming',
 '215Days',
 '3doorsdown',
 '4AD',
 '4KD',
 '50 Cent',
 '5SOS',
 '60s70sVintageRock',
 '77GhettoD',
 'A Flock of Seagulls',
 'ABBA',
 'ABKCOVEVO',
 'AC/DC',
 'AI Videoworks',
 'AK47bandit',
 'AL NUCAHC',
 'Aaron Neville',
 'Ace of Base',
 'Adele',
 'Aerosmith',
 'Akon',
 'Al Green',
 'Alanis Morissette',
 'Alice Deejay',
 'Alicia Keys',
 'All Time Low',
 'Allman Brothers Band',
 'Amy Winehouse',
 'AnalogAnarchy99',
 'Andres Trevino',
 'Annie Lennox',
 'Antonio',
 'Aqua',
 'Aretha Franklin',
 'Ariana Grande',
 'Atlantic Records',
 'Avicii',
 'Avril Lavigne',
 'BBC Music',
 'BBC Politics',
 'BLUE OYSTER CULT',
 'BOSTON',
 'BT Music Videos',
 'Backstreet Boys',
 'Bad Boy Entertainment',
 'BeastieBoys',
 'Beat-Club',
 'Beck',
 'Belinda Carlisle',
 'Bell Biv Devoe',
 'Ben E. King',
 'Berlin',
 'Bert',
 'Bertha0815007',
 'Bertrand Vandemeulebroucke',
 'Betaman31252',
 'Beyoncé',
 'Big Pun',
 'Big Star',
 'Bill Withers',
 'Billie Eilish',
 'Billy Joel',


In [16]:
df_music_videos_dataset["uploader"].dropna().nunique()

673

In [17]:
df_music_videos_dataset["uploader"].dropna().value_counts().sort_index()

uploader
*NSYNC                 2
05vs1                  1
1984dreaming           1
215Days                1
3doorsdown             2
                      ..
worldsgreatestmusic    1
yeahyeahyeahsmusic     1
yogadoors              1
АП                     1
Александр Кирсанов     1
Name: count, Length: 673, dtype: int64

## Export `df_music_videos_dataset` into NDJSON, TSV and XLSX formats

In [18]:
df_music_videos_dataset.to_json('corpus/00_sources/music_videos_dataset.ndjson', orient='records', lines=True)
df_music_videos_dataset.to_csv('corpus/00_sources/music_videos_dataset.tsv', sep='	', index=False)
df_music_videos_dataset.to_excel('corpus/00_sources/music_videos_dataset.xlsx', index=False)